# NTO AI 2025/2026 Final — strong notebook: ALS + item2item KNN + CatBoostRanker ensemble

Что делает ноутбук:

1. Строит **псевдо-валидацию**, максимально похожую на реальную задачу: скрываются только **новые позитивные пары** `(user_id, edition_id)` внутри incident-like окна.
2. Генерирует кандидатов несколькими сильными способами:
   - **ALS full**
   - **ALS recent**
   - **item x item KNN** в латентном пространстве ALS
   - **item x item sparse KNN** по user-item матрице
   - популярные издания по **author / book / genre**
   - **demographic popularity** для cold users
   - глобальная recent popularity как safe fallback
3. Собирает сильные фичи:
   - source-level признаки кандидата
   - user / item агрегаты
   - affinity по author / book / language / publisher
   - ALS-based признаки
   - KNN aggregate признаки
   - тренды и recentness
4. Учит **ансамбль CatBoostRanker**:
   - `YetiRankPairwise`
   - `QuerySoftMax`
5. Делает blending по локальной валидации, retrain на всех pseudo folds и пишет `submission.csv`.

Ноутбук написан так, чтобы **скомпилироваться и работать в Kaggle** без внешнего baseline-репозитория.  
Если пакет `implicit` доступен — используется быстрый ALS. Если нет, включается fallback ALS на `numpy/scipy/numba`.


In [ ]:
from __future__ import annotations

import gc
import math
import os
import random
import warnings
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 200)


DEFAULT_WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")


@dataclass
class CFG:
    seed: int = 42
    use_gpu: bool = Path("/proc/driver/nvidia/version").exists()
    debug: bool = False
    debug_n_users: int | None = None

    # Data / cache
    cache_dir: Path = DEFAULT_WORKDIR / "nto_cache"
    output_dir: Path = DEFAULT_WORKDIR

    # Temporal CV
    fold_days: int = 30
    n_pseudo_folds: int = 3           # 2 train folds + 1 validation fold by default
    hide_rate: float = 0.20

    # Interaction weights
    read_weight: float = 2.4
    wishlist_weight: float = 1.0
    full_tau_days: float = 90.0
    recent_tau_days: float = 21.0

    # Candidate generation
    als_full_topk: int = 180
    als_recent_topk: int = 140
    latent_knn_neighbors: int = 90
    sparse_knn_neighbors: int = 85
    read_knn_neighbors: int = 70
    top_authors_per_user: int = 6
    top_books_per_user: int = 8
    top_genres_per_user: int = 6
    top_author_items: int = 20
    top_book_items: int = 12
    top_genre_items: int = 20
    top_demo_items: int = 20
    top_global_items: int = 30
    recent_seed_items: int = 6
    heavy_seed_items: int = 6
    max_candidates_per_user: int = 420

    # ALS
    als_full_factors: int = 128
    als_recent_factors: int = 96
    als_iterations: int = 18
    als_recent_iterations: int = 14
    als_regularization: float = 0.06

    # Fallback ALS (when implicit package is unavailable)
    fallback_als_full_factors: int = 48
    fallback_als_recent_factors: int = 32
    fallback_als_iterations: int = 8
    fallback_cg: bool = False

    # Ranking dataset
    max_negatives_per_user: int = 220
    min_group_size_for_train: int = 10

    # Models
    cb_iter_a: int = 1600
    cb_iter_b: int = 1800
    cb_lr_a: float = 0.045
    cb_lr_b: float = 0.040

    # Misc
    batch_size_reco: int = 128
    save_intermediate: bool = True


cfg = CFG()
cfg.cache_dir.mkdir(parents=True, exist_ok=True)
cfg.output_dir.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(cfg.seed)
print(cfg)


In [ ]:
REQUIRED_FILES = [
    "interactions.csv",
    "targets.csv",
    "editions.csv",
    "users.csv",
    "authors.csv",
    "genres.csv",
    "book_genres.csv",
]


def find_data_root(required_files: list[str] = REQUIRED_FILES) -> Path:
    candidates = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
        Path("/mnt/data"),
        Path("."),
    ]
    direct_hits: list[Path] = []

    for root in candidates:
        if not root.exists():
            continue
        if all((root / f).exists() for f in required_files):
            return root
        for p in root.rglob(required_files[0]):
            base = p.parent
            if all((base / f).exists() for f in required_files):
                direct_hits.append(base)

    if not direct_hits:
        raise FileNotFoundError(
            "Не удалось найти папку с данными. "
            "Ожидаются файлы: "
            + ", ".join(required_files)
        )
    direct_hits = sorted(set(direct_hits))
    return direct_hits[0]


DATA_ROOT = find_data_root()
print("DATA_ROOT:", DATA_ROOT)


def make_age_bucket(series: pd.Series) -> pd.Series:
    bins = [-1, 17, 24, 34, 44, 54, 120]
    labels = [0, 1, 2, 3, 4, 5]
    out = pd.cut(series.fillna(-1), bins=bins, labels=labels)
    out = out.astype("float").fillna(-1).astype("int16")
    return out


def load_tables(data_root: Path) -> dict[str, pd.DataFrame]:
    interactions = pd.read_csv(
        data_root / "interactions.csv",
        low_memory=False,
    )
    targets = pd.read_csv(data_root / "targets.csv")
    editions = pd.read_csv(data_root / "editions.csv", low_memory=False)
    users = pd.read_csv(data_root / "users.csv")
    authors = pd.read_csv(data_root / "authors.csv", low_memory=False)
    genres = pd.read_csv(data_root / "genres.csv", low_memory=False)
    book_genres = pd.read_csv(data_root / "book_genres.csv")

    interactions["event_ts"] = pd.to_datetime(interactions["event_ts"])
    interactions = interactions[interactions["event_type"].isin([1, 2])].copy()

    # Safe numeric casts
    for col in ["user_id", "edition_id"]:
        interactions[col] = interactions[col].astype("int64")
    interactions["event_type"] = interactions["event_type"].astype("int8")
    if "rating" in interactions.columns:
        interactions["rating"] = pd.to_numeric(interactions["rating"], errors="coerce").astype("float32")

    users["user_id"] = users["user_id"].astype("int64")
    users["gender"] = pd.to_numeric(users["gender"], errors="coerce").fillna(-1).astype("int16")
    users["age"] = pd.to_numeric(users["age"], errors="coerce").astype("float32")
    users["age_bucket"] = make_age_bucket(users["age"])

    targets["user_id"] = targets["user_id"].astype("int64")

    editions["edition_id"] = editions["edition_id"].astype("int64")
    editions["book_id"] = pd.to_numeric(editions["book_id"], errors="coerce").fillna(-1).astype("int64")
    editions["author_id"] = pd.to_numeric(editions["author_id"], errors="coerce").fillna(-1).astype("int64")
    editions["publication_year"] = pd.to_numeric(editions["publication_year"], errors="coerce").astype("float32")
    editions["age_restriction"] = pd.to_numeric(editions["age_restriction"], errors="coerce").fillna(-1).astype("int16")
    editions["language_id"] = pd.to_numeric(editions["language_id"], errors="coerce").fillna(-1).astype("int64")
    editions["publisher_id"] = pd.to_numeric(editions["publisher_id"], errors="coerce").fillna(-1).astype("int64")

    for col in ["title", "description"]:
        if col not in editions.columns:
            editions[col] = ""
        editions[col] = editions[col].fillna("").astype(str)

    editions["title_len"] = editions["title"].str.len().astype("int32")
    editions["desc_len"] = editions["description"].str.len().astype("int32")

    authors["author_id"] = authors["author_id"].astype("int64")
    if "author_name" not in authors.columns:
        authors["author_name"] = ""
    authors["author_name"] = authors["author_name"].fillna("").astype(str)

    genres["genre_id"] = genres["genre_id"].astype("int64")
    genres["genre_name"] = genres["genre_name"].fillna("").astype(str)

    book_genres["book_id"] = pd.to_numeric(book_genres["book_id"], errors="coerce").fillna(-1).astype("int64")
    book_genres["genre_id"] = pd.to_numeric(book_genres["genre_id"], errors="coerce").fillna(-1).astype("int64")

    interactions = interactions.sort_values("event_ts").reset_index(drop=True)

    return {
        "interactions": interactions,
        "targets": targets,
        "editions": editions,
        "users": users,
        "authors": authors,
        "genres": genres,
        "book_genres": book_genres,
    }


tables = load_tables(DATA_ROOT)
for k, v in tables.items():
    print(f"{k:12s} -> {v.shape}")

interactions = tables["interactions"]
targets = tables["targets"]
editions = tables["editions"]
users = tables["users"]
authors = tables["authors"]
genres = tables["genres"]
book_genres = tables["book_genres"]

if cfg.debug and cfg.debug_n_users is not None:
    keep_users = set(interactions["user_id"].drop_duplicates().head(cfg.debug_n_users).tolist())
    keep_users |= set(targets["user_id"].head(max(1000, cfg.debug_n_users)).tolist())
    interactions = interactions[interactions["user_id"].isin(keep_users)].copy()
    users = users[users["user_id"].isin(keep_users)].copy()
    targets = targets[targets["user_id"].isin(keep_users)].copy()
    print("DEBUG mode shapes:", interactions.shape, targets.shape)


In [ ]:
def build_edition_meta(
    editions: pd.DataFrame,
    authors: pd.DataFrame,
    book_genres: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    genre_lists = (
        book_genres.groupby("book_id")["genre_id"]
        .agg(lambda x: tuple(sorted(set(int(v) for v in x if pd.notna(v)))))
        .reset_index()
    )
    genre_lists["genre_count"] = genre_lists["genre_id"].apply(len).astype("int16")
    genre_lists = genre_lists.rename(columns={"genre_id": "genre_ids"})

    meta = editions.merge(authors[["author_id", "author_name"]], on="author_id", how="left")
    meta = meta.merge(genre_lists, on="book_id", how="left")

    meta["author_name"] = meta["author_name"].fillna("")
    meta["genre_ids"] = meta["genre_ids"].apply(lambda x: x if isinstance(x, tuple) else tuple())
    meta["genre_count"] = meta["genre_count"].fillna(0).astype("int16")
    meta["publication_year"] = meta["publication_year"].astype("float32")
    meta["item_age_proxy"] = (2025 - meta["publication_year"]).clip(-100, 300).fillna(0).astype("float32")
    meta["title_desc_len"] = (meta["title_len"] + meta["desc_len"]).astype("int32")

    edition_to_book = meta.set_index("edition_id")["book_id"].to_dict()
    edition_genres = meta[["edition_id", "book_id"]].merge(book_genres, on="book_id", how="left")
    edition_genres["genre_id"] = edition_genres["genre_id"].fillna(-1).astype("int64")
    edition_genres = edition_genres[edition_genres["genre_id"] >= 0].copy()

    return meta, edition_genres


edition_meta, edition_genres = build_edition_meta(editions, authors, book_genres)

print("edition_meta:", edition_meta.shape)
print("edition_genres:", edition_genres.shape)

display(edition_meta.head(2))
display(edition_genres.head(2))


In [ ]:
TOP_K = 20


def validate_submission_rows(
    submission_rows: list[dict[str, Any]],
    target_users: set[int],
    top_k: int = TOP_K,
) -> tuple[bool, list[str]]:
    errors: list[str] = []
    by_user: dict[int, list[tuple[int, int]]] = defaultdict(list)

    for row in submission_rows:
        try:
            user_id = int(row.get("user_id"))
            edition_id = int(row.get("edition_id"))
            rank = int(row.get("rank"))
        except Exception:
            errors.append("Rows must contain valid integer user_id / edition_id / rank")
            continue

        if rank < 1 or rank > top_k:
            errors.append(f"Rank out of range 1..{top_k} for user {user_id}: {rank}")
        by_user[user_id].append((rank, edition_id))

    missing = target_users - set(by_user.keys())
    extra = set(by_user.keys()) - target_users
    if missing:
        errors.append(f"Missing target users: {len(missing)}")
    if extra:
        errors.append(f"Unexpected users in submission: {len(extra)}")

    expected_ranks = set(range(1, top_k + 1))
    for user_id, rows in by_user.items():
        if len(rows) != top_k:
            errors.append(f"User {user_id}: expected {top_k} rows, got {len(rows)}")
            continue
        ranks = [r for r, _ in rows]
        items = [i for _, i in rows]
        if set(ranks) != expected_ranks or len(set(ranks)) != top_k:
            errors.append(f"User {user_id}: ranks must be unique 1..{top_k}")
        if len(set(items)) != top_k:
            errors.append(f"User {user_id}: edition_id must be unique")

    return len(errors) == 0, errors


def ndcg_at_k(predicted: list[int], relevant: set[int], k: int = TOP_K) -> float:
    if not relevant:
        return 0.0
    dcg = 0.0
    for idx, item in enumerate(predicted[:k], start=1):
        rel = 1.0 if item in relevant else 0.0
        dcg += rel / math.log2(idx + 1)
    idcg = sum(1.0 / math.log2(idx + 1) for idx in range(1, min(len(relevant), k) + 1))
    return dcg / idcg if idcg > 0.0 else 0.0


def evaluate_scored_candidates(
    scored_df: pd.DataFrame,
    hidden_pairs: pd.DataFrame,
    score_col: str,
    k: int = TOP_K,
) -> float:
    rel_map = (
        hidden_pairs.groupby("user_id")["edition_id"]
        .agg(lambda x: set(int(v) for v in x.tolist()))
        .to_dict()
    )
    total = 0.0
    users_eval = 0

    for user_id, grp in scored_df.sort_values(["user_id", score_col], ascending=[True, False]).groupby("user_id"):
        relevant = rel_map.get(int(user_id), set())
        if not relevant:
            continue
        predicted = grp["edition_id"].head(k).astype(int).tolist()
        total += ndcg_at_k(predicted, relevant, k=k)
        users_eval += 1

    return total / max(users_eval, 1)


def candidate_recall_at_k(
    candidate_df: pd.DataFrame,
    hidden_pairs: pd.DataFrame,
    pre_col: str = "pre_score",
    k: int = 200,
) -> float:
    rel = hidden_pairs.assign(label=1)
    topk = (
        candidate_df.sort_values(["user_id", pre_col], ascending=[True, False])
        .groupby("user_id")
        .head(k)[["user_id", "edition_id"]]
        .drop_duplicates()
    )
    merged = rel.merge(topk, on=["user_id", "edition_id"], how="left", indicator=True)
    return float((merged["_merge"] == "both").mean()) if len(merged) else 0.0


In [ ]:
@dataclass
class FoldSpec:
    name: str
    window_start: pd.Timestamp
    window_end: pd.Timestamp
    seed: int


def make_temporal_folds(interactions: pd.DataFrame, cfg: CFG) -> list[FoldSpec]:
    max_ts = interactions["event_ts"].max().normalize() + pd.Timedelta(days=1)
    min_ts = interactions["event_ts"].min().normalize()
    folds: list[FoldSpec] = []

    for i in range(cfg.n_pseudo_folds):
        window_end = max_ts - pd.Timedelta(days=cfg.fold_days * (cfg.n_pseudo_folds - 1 - i))
        window_start = window_end - pd.Timedelta(days=cfg.fold_days)
        if window_start <= min_ts:
            continue
        folds.append(
            FoldSpec(
                name=f"fold_{i}",
                window_start=window_start,
                window_end=window_end,
                seed=cfg.seed + 100 + i,
            )
        )
    return folds


def aggregate_window_pairs(window_df: pd.DataFrame) -> pd.DataFrame:
    if window_df.empty:
        return pd.DataFrame(
            columns=[
                "user_id", "edition_id", "pair_last_ts", "pair_count",
                "has_read", "has_wishlist", "max_rating"
            ]
        )

    agg = (
        window_df.groupby(["user_id", "edition_id"], as_index=False)
        .agg(
            pair_last_ts=("event_ts", "max"),
            pair_count=("event_ts", "size"),
            has_read=("event_type", lambda x: int((x == 2).any())),
            has_wishlist=("event_type", lambda x: int((x == 1).any())),
            max_rating=("rating", "max"),
        )
    )
    return agg


def weighted_mask_incident_pairs(
    pair_df: pd.DataFrame,
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
    hide_rate: float,
    seed: int,
) -> pd.DataFrame:
    if pair_df.empty:
        return pair_df.copy()

    rng = np.random.default_rng(seed)
    out_idx: list[int] = []

    pair_df = pair_df.copy()
    total_seconds = max((window_end - window_start).total_seconds(), 1.0)
    pair_df["within_window_frac"] = (
        (pair_df["pair_last_ts"] - window_start).dt.total_seconds() / total_seconds
    ).clip(0.0, 1.0)

    pair_df["pair_weight"] = (
        1.0
        + 0.45 * pair_df["has_read"].astype(float)
        + 0.20 * pair_df["has_wishlist"].astype(float)
        + 0.25 * np.log1p(pair_df["pair_count"].astype(float))
        + 0.35 * pair_df["within_window_frac"].astype(float)
        + 0.10 * pair_df["max_rating"].fillna(0).ge(4).astype(float)
    )

    for user_id, grp in pair_df.groupby("user_id", sort=False):
        n = len(grp)
        if n == 0:
            continue

        user_noise = float(rng.lognormal(mean=0.0, sigma=0.35))
        user_rate = float(np.clip(hide_rate * user_noise, 0.05, 0.60))
        n_hide = int(round(n * user_rate))

        if n_hide == 0 and n >= 3 and rng.random() < 0.35:
            n_hide = 1
        n_hide = max(0, min(n_hide, n))

        if n_hide == 0:
            continue

        p = grp["pair_weight"].to_numpy(dtype=np.float64)
        p = p / p.sum()
        chosen = rng.choice(grp.index.to_numpy(), size=n_hide, replace=False, p=p)
        out_idx.extend(chosen.tolist())

    return pair_df.loc[out_idx, ["user_id", "edition_id"]].drop_duplicates().reset_index(drop=True)


def build_masked_fold(
    interactions: pd.DataFrame,
    fold: FoldSpec,
    cfg: CFG,
) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
    history_before = interactions[interactions["event_ts"] < fold.window_start][["user_id", "edition_id"]].drop_duplicates()
    incident = interactions[
        (interactions["event_ts"] >= fold.window_start) &
        (interactions["event_ts"] < fold.window_end)
    ].copy()

    # Важный момент:
    # скрываем только пары, которых НЕ было до начала окна,
    # иначе после скрытия они всё равно останутся observed на pair-level.
    eligible = incident.merge(
        history_before.assign(_seen_before=1),
        on=["user_id", "edition_id"],
        how="left",
    )
    eligible = eligible[eligible["_seen_before"].isna()].drop(columns="_seen_before")

    eligible_pairs = aggregate_window_pairs(eligible)
    hidden_pairs = weighted_mask_incident_pairs(
        eligible_pairs,
        fold.window_start,
        fold.window_end,
        hide_rate=cfg.hide_rate,
        seed=fold.seed,
    )

    observed = interactions[interactions["event_ts"] < fold.window_end].copy()
    if not hidden_pairs.empty:
        observed = observed.merge(
            hidden_pairs.assign(_hide=1),
            on=["user_id", "edition_id"],
            how="left",
        )
        observed = observed[observed["_hide"].isna()].drop(columns="_hide")

    fold_users = np.array(sorted(hidden_pairs["user_id"].unique()), dtype=np.int64)
    return observed.reset_index(drop=True), hidden_pairs.reset_index(drop=True), fold_users


folds = make_temporal_folds(interactions, cfg)
for f in folds:
    print(f.name, f.window_start, "->", f.window_end)


In [ ]:
def add_interaction_weights(df: pd.DataFrame, ref_ts: pd.Timestamp, cfg: CFG) -> pd.DataFrame:
    df = df.copy()
    age_days = ((ref_ts - df["event_ts"]).dt.total_seconds() / 86400.0).clip(lower=0.0)
    evt_w = np.where(df["event_type"].eq(2), cfg.read_weight, cfg.wishlist_weight).astype("float32")
    rating_adj = np.where(
        df["rating"].notna(),
        1.0 + 0.08 * (df["rating"].fillna(0).to_numpy(dtype=np.float32) - 3.0),
        1.0,
    ).astype("float32")
    rating_adj = np.clip(rating_adj, 0.65, 1.35)

    df["is_read"] = (df["event_type"] == 2).astype("int8")
    df["is_wishlist"] = (df["event_type"] == 1).astype("int8")
    df["ui_weight"] = evt_w * (0.35 + 0.65 * np.exp(-age_days / cfg.full_tau_days)) * rating_adj
    df["ui_recent_weight"] = evt_w * np.exp(-age_days / cfg.recent_tau_days) * rating_adj
    df["age_days"] = age_days.astype("float32")
    return df


def build_pair_aggregates(observed: pd.DataFrame, ref_ts: pd.Timestamp, cfg: CFG) -> pd.DataFrame:
    obs = add_interaction_weights(observed, ref_ts, cfg)

    pair_df = (
        obs.groupby(["user_id", "edition_id"], as_index=False)
        .agg(
            ui_events=("event_ts", "size"),
            ui_reads=("is_read", "sum"),
            ui_wish=("is_wishlist", "sum"),
            ui_weight=("ui_weight", "sum"),
            ui_recent_weight=("ui_recent_weight", "sum"),
            ui_last_ts=("event_ts", "max"),
            ui_first_ts=("event_ts", "min"),
            ui_max_rating=("rating", "max"),
            ui_mean_rating=("rating", "mean"),
        )
    )
    pair_df["ui_last_days"] = ((ref_ts - pair_df["ui_last_ts"]).dt.total_seconds() / 86400.0).astype("float32")
    pair_df["ui_first_days"] = ((ref_ts - pair_df["ui_first_ts"]).dt.total_seconds() / 86400.0).astype("float32")
    pair_df["ui_log_events"] = np.log1p(pair_df["ui_events"]).astype("float32")
    return pair_df


def build_user_features(
    obs_weighted: pd.DataFrame,
    pair_df: pd.DataFrame,
    users: pd.DataFrame,
    ref_ts: pd.Timestamp,
) -> pd.DataFrame:
    base = (
        pair_df.groupby("user_id", as_index=False)
        .agg(
            user_seen_items=("edition_id", "nunique"),
            user_events=("ui_events", "sum"),
            user_reads=("ui_reads", "sum"),
            user_wish=("ui_wish", "sum"),
            user_weight=("ui_weight", "sum"),
            user_recent_weight=("ui_recent_weight", "sum"),
            user_last_days=("ui_last_days", "min"),
            user_avg_pair_rating=("ui_mean_rating", "mean"),
        )
    )

    for days in [7, 30, 90]:
        recent = obs_weighted[obs_weighted["event_ts"] >= ref_ts - pd.Timedelta(days=days)]
        if recent.empty:
            tmp = pd.DataFrame(columns=["user_id"])
        else:
            tmp = (
                recent.groupby("user_id", as_index=False)
                .agg(
                    **{
                        f"user_events_{days}d": ("event_ts", "size"),
                        f"user_items_{days}d": ("edition_id", "nunique"),
                        f"user_weight_{days}d": ("ui_weight", "sum"),
                    }
                )
            )
        base = base.merge(tmp, on="user_id", how="left")

    base = base.merge(users[["user_id", "gender", "age", "age_bucket"]], on="user_id", how="left")

    fill_zero_cols = [c for c in base.columns if c.startswith(("user_events_", "user_items_", "user_weight_"))]
    for col in fill_zero_cols:
        base[col] = base[col].fillna(0)

    base["user_seen_items"] = base["user_seen_items"].fillna(0).astype("int32")
    base["user_events"] = base["user_events"].fillna(0).astype("int32")
    base["user_reads"] = base["user_reads"].fillna(0).astype("int32")
    base["user_wish"] = base["user_wish"].fillna(0).astype("int32")
    base["user_weight"] = base["user_weight"].fillna(0).astype("float32")
    base["user_recent_weight"] = base["user_recent_weight"].fillna(0).astype("float32")
    base["user_last_days"] = base["user_last_days"].fillna(9999).astype("float32")
    base["user_avg_pair_rating"] = base["user_avg_pair_rating"].fillna(0).astype("float32")
    base["user_read_share"] = (
        base["user_reads"] / np.maximum(base["user_events"], 1)
    ).astype("float32")
    base["user_recent_to_all_ratio"] = (
        base["user_recent_weight"] / np.maximum(base["user_weight"], 1e-6)
    ).astype("float32")
    u_w7 = base["user_weight_7d"] if "user_weight_7d" in base.columns else 0.0
    u_w30 = base["user_weight_30d"] if "user_weight_30d" in base.columns else 0.0
    base["user_hot_ratio"] = (u_w7 / np.maximum(u_w30, 1e-6)).astype("float32")
    base["gender"] = base["gender"].fillna(-1).astype("int16")
    base["age"] = base["age"].fillna(-1).astype("float32")
    base["age_bucket"] = base["age_bucket"].fillna(-1).astype("int16")
    return base


def build_item_features(
    obs_weighted: pd.DataFrame,
    pair_df: pd.DataFrame,
    edition_meta: pd.DataFrame,
    ref_ts: pd.Timestamp,
) -> pd.DataFrame:
    base = (
        pair_df.groupby("edition_id", as_index=False)
        .agg(
            item_users=("user_id", "nunique"),
            item_events=("ui_events", "sum"),
            item_reads=("ui_reads", "sum"),
            item_wish=("ui_wish", "sum"),
            item_weight=("ui_weight", "sum"),
            item_recent_weight=("ui_recent_weight", "sum"),
            item_last_days=("ui_last_days", "min"),
            item_avg_pair_rating=("ui_mean_rating", "mean"),
        )
    )

    for days in [7, 30, 90]:
        recent = obs_weighted[obs_weighted["event_ts"] >= ref_ts - pd.Timedelta(days=days)]
        if recent.empty:
            tmp = pd.DataFrame(columns=["edition_id"])
        else:
            tmp = (
                recent.groupby("edition_id", as_index=False)
                .agg(
                    **{
                        f"item_events_{days}d": ("event_ts", "size"),
                        f"item_users_{days}d": ("user_id", "nunique"),
                        f"item_weight_{days}d": ("ui_weight", "sum"),
                    }
                )
            )
        base = base.merge(tmp, on="edition_id", how="left")

    base = base.merge(
        edition_meta[
            [
                "edition_id", "book_id", "author_id", "publication_year",
                "age_restriction", "language_id", "publisher_id", "title_len",
                "desc_len", "genre_count", "item_age_proxy", "title_desc_len"
            ]
        ],
        on="edition_id",
        how="left",
    )

    fill_zero_cols = [c for c in base.columns if c.startswith(("item_events_", "item_users_", "item_weight_"))]
    for col in fill_zero_cols:
        base[col] = base[col].fillna(0)

    base["item_users"] = base["item_users"].fillna(0).astype("int32")
    base["item_events"] = base["item_events"].fillna(0).astype("int32")
    base["item_reads"] = base["item_reads"].fillna(0).astype("int32")
    base["item_wish"] = base["item_wish"].fillna(0).astype("int32")
    base["item_weight"] = base["item_weight"].fillna(0).astype("float32")
    base["item_recent_weight"] = base["item_recent_weight"].fillna(0).astype("float32")
    base["item_last_days"] = base["item_last_days"].fillna(9999).astype("float32")
    base["item_avg_pair_rating"] = base["item_avg_pair_rating"].fillna(0).astype("float32")
    base["item_read_share"] = (base["item_reads"] / np.maximum(base["item_events"], 1)).astype("float32")
    base["item_recent_to_all_ratio"] = (base["item_recent_weight"] / np.maximum(base["item_weight"], 1e-6)).astype("float32")
    i_w7 = base["item_weight_7d"] if "item_weight_7d" in base.columns else 0.0
    i_w30 = base["item_weight_30d"] if "item_weight_30d" in base.columns else 0.0
    i_w90 = base["item_weight_90d"] if "item_weight_90d" in base.columns else 0.0
    base["item_hot_ratio"] = (i_w7 / np.maximum(i_w30, 1e-6)).astype("float32")
    base["item_trend_ratio"] = ((i_w30 + 1.0) / (i_w90 + 1.0)).astype("float32")
    base["item_trend_delta"] = (i_w7 - i_w30 / 4.0).astype("float32")

    for col in ["book_id", "author_id", "language_id", "publisher_id"]:
        base[col] = base[col].fillna(-1).astype("int64")
    for col in ["publication_year", "item_age_proxy"]:
        base[col] = base[col].fillna(0).astype("float32")
    for col in ["age_restriction", "title_len", "desc_len", "genre_count", "title_desc_len"]:
        base[col] = base[col].fillna(0).astype("int32")

    base["item_pop_rank"] = base["item_recent_weight"].rank(method="dense", ascending=False).astype("float32")
    return base


def build_affinity_tables(
    pair_df: pd.DataFrame,
    edition_meta: pd.DataFrame,
    edition_genres: pd.DataFrame,
) -> dict[str, pd.DataFrame]:
    pair_meta = pair_df.merge(
        edition_meta[["edition_id", "author_id", "book_id", "language_id", "publisher_id"]],
        on="edition_id",
        how="left",
    )

    def _agg_by(entity_col: str, prefix: str) -> pd.DataFrame:
        out = (
            pair_meta.groupby(["user_id", entity_col], as_index=False)
            .agg(
                **{
                    f"{prefix}_weight": ("ui_weight", "sum"),
                    f"{prefix}_recent_weight": ("ui_recent_weight", "sum"),
                    f"{prefix}_items": ("edition_id", "nunique"),
                    f"{prefix}_last_days": ("ui_last_days", "min"),
                }
            )
        )
        out[f"{prefix}_rank"] = (
            out.groupby("user_id")[f"{prefix}_recent_weight"]
            .rank(method="dense", ascending=False)
            .astype("float32")
        )
        return out

    user_author = _agg_by("author_id", "ua")
    user_book = _agg_by("book_id", "ub")
    user_lang = _agg_by("language_id", "ul")
    user_pub = _agg_by("publisher_id", "up")

    pair_genres = pair_df[["user_id", "edition_id", "ui_weight", "ui_recent_weight", "ui_last_days"]].merge(
        edition_genres, on="edition_id", how="left"
    )
    pair_genres = pair_genres[pair_genres["genre_id"].notna()].copy()
    pair_genres["genre_id"] = pair_genres["genre_id"].astype("int64")

    user_genre = (
        pair_genres.groupby(["user_id", "genre_id"], as_index=False)
        .agg(
            ug_weight=("ui_weight", "sum"),
            ug_recent_weight=("ui_recent_weight", "sum"),
            ug_items=("edition_id", "nunique"),
            ug_last_days=("ui_last_days", "min"),
        )
    )
    user_genre["ug_rank"] = (
        user_genre.groupby("user_id")["ug_recent_weight"]
        .rank(method="dense", ascending=False)
        .astype("float32")
    )

    return {
        "user_author": user_author,
        "user_book": user_book,
        "user_lang": user_lang,
        "user_pub": user_pub,
        "user_genre": user_genre,
        "pair_meta": pair_meta,
    }


def build_seed_items(pair_df: pd.DataFrame, cfg: CFG) -> pd.DataFrame:
    recent = (
        pair_df.sort_values(["user_id", "ui_last_days", "ui_recent_weight"], ascending=[True, True, False])
        .groupby("user_id")
        .head(cfg.recent_seed_items)
        .copy()
    )
    recent["seed_source"] = "recent"
    recent["seed_rank"] = recent.groupby("user_id").cumcount() + 1
    recent["seed_weight"] = (
        (1.0 / (1.0 + recent["ui_last_days"] / 7.0))
        * (1.0 + 0.15 * recent["ui_reads"])
    ).astype("float32")

    heavy = (
        pair_df.sort_values(["user_id", "ui_recent_weight", "ui_weight"], ascending=[True, False, False])
        .groupby("user_id")
        .head(cfg.heavy_seed_items)
        .copy()
    )
    heavy["seed_source"] = "heavy"
    heavy["seed_rank"] = heavy.groupby("user_id").cumcount() + 1
    heavy["seed_weight"] = (
        np.log1p(heavy["ui_recent_weight"] + heavy["ui_weight"])
    ).astype("float32")

    seeds = pd.concat([recent, heavy], ignore_index=True)
    seeds = (
        seeds.groupby(["user_id", "edition_id"], as_index=False)
        .agg(
            seed_weight=("seed_weight", "max"),
            seed_rank=("seed_rank", "min"),
            ui_last_days=("ui_last_days", "min"),
            ui_reads=("ui_reads", "max"),
        )
    )
    return seeds


def build_popularity_tables(
    pair_df: pd.DataFrame,
    affinity: dict[str, pd.DataFrame],
    users: pd.DataFrame,
    cfg: CFG,
) -> dict[str, pd.DataFrame]:
    item_pop_score = pair_df.groupby("edition_id", as_index=False).agg(
        pop_score=("ui_recent_weight", "sum"),
        pop_score_full=("ui_weight", "sum"),
        pop_users=("user_id", "nunique"),
    )
    item_pop_score["pop_rank"] = item_pop_score["pop_score"].rank(method="dense", ascending=False)
    global_pop = item_pop_score.sort_values(["pop_score", "pop_score_full"], ascending=False).head(cfg.top_global_items).copy()
    recent_pop = global_pop.copy()
    recent_pop = recent_pop.rename(columns={"pop_rank": "recent_pop_rank", "pop_score": "recent_pop_score"})

    user_author = affinity["user_author"].copy()
    user_book = affinity["user_book"].copy()
    user_genre = affinity["user_genre"].copy()

    author_item_pop = affinity["pair_meta"].groupby(["author_id", "edition_id"], as_index=False).agg(
        author_item_score=("ui_recent_weight", "sum")
    )
    author_item_pop["author_item_rank"] = (
        author_item_pop.groupby("author_id")["author_item_score"]
        .rank(method="dense", ascending=False)
        .astype("float32")
    )
    author_item_pop = author_item_pop[author_item_pop["author_item_rank"] <= cfg.top_author_items].copy()

    book_item_pop = affinity["pair_meta"].groupby(["book_id", "edition_id"], as_index=False).agg(
        book_item_score=("ui_recent_weight", "sum")
    )
    book_item_pop["book_item_rank"] = (
        book_item_pop.groupby("book_id")["book_item_score"]
        .rank(method="dense", ascending=False)
        .astype("float32")
    )
    book_item_pop = book_item_pop[book_item_pop["book_item_rank"] <= cfg.top_book_items].copy()

    pair_genres = affinity["pair_meta"][["user_id", "edition_id", "ui_recent_weight"]].merge(
        edition_genres, on="edition_id", how="left"
    )
    genre_item_pop = pair_genres.groupby(["genre_id", "edition_id"], as_index=False).agg(
        genre_item_score=("ui_recent_weight", "sum")
    )
    genre_item_pop["genre_item_rank"] = (
        genre_item_pop.groupby("genre_id")["genre_item_score"]
        .rank(method="dense", ascending=False)
        .astype("float32")
    )
    genre_item_pop = genre_item_pop[genre_item_pop["genre_item_rank"] <= cfg.top_genre_items].copy()

    obs_users = pair_df[["user_id"]].drop_duplicates().merge(users[["user_id", "gender", "age_bucket"]], on="user_id", how="left")
    user_item_recent = pair_df.merge(obs_users, on="user_id", how="left")

    demo_pop = (
        user_item_recent.groupby(["gender", "age_bucket", "edition_id"], as_index=False)
        .agg(demo_score=("ui_recent_weight", "sum"))
    )
    demo_pop["demo_rank"] = (
        demo_pop.groupby(["gender", "age_bucket"])["demo_score"]
        .rank(method="dense", ascending=False)
        .astype("float32")
    )
    demo_pop = demo_pop[demo_pop["demo_rank"] <= cfg.top_demo_items].copy()

    gender_pop = (
        user_item_recent.groupby(["gender", "edition_id"], as_index=False)
        .agg(gender_score=("ui_recent_weight", "sum"))
    )
    gender_pop["gender_rank"] = (
        gender_pop.groupby("gender")["gender_score"]
        .rank(method="dense", ascending=False)
        .astype("float32")
    )
    gender_pop = gender_pop[gender_pop["gender_rank"] <= cfg.top_demo_items].copy()

    return {
        "global_pop": global_pop,
        "recent_pop": recent_pop,
        "author_item_pop": author_item_pop,
        "book_item_pop": book_item_pop,
        "genre_item_pop": genre_item_pop,
        "demo_pop": demo_pop,
        "gender_pop": gender_pop,
        "user_author_top": user_author[user_author["ua_rank"] <= cfg.top_authors_per_user].copy(),
        "user_book_top": user_book[user_book["ub_rank"] <= cfg.top_books_per_user].copy(),
        "user_genre_top": user_genre[user_genre["ug_rank"] <= cfg.top_genres_per_user].copy(),
    }


In [ ]:
try:
    import implicit  # type: ignore
    HAS_IMPLICIT = True
except Exception:
    HAS_IMPLICIT = False

print("implicit available:", HAS_IMPLICIT)


def build_interaction_matrix(
    pair_df: pd.DataFrame,
    value_col: str,
) -> tuple[sparse.csr_matrix, dict[int, int], np.ndarray]:
    user_ids = np.sort(pair_df["user_id"].unique())
    item_ids = np.sort(pair_df["edition_id"].unique())
    user_to_idx = {int(u): i for i, u in enumerate(user_ids)}
    item_to_idx = {int(it): i for i, it in enumerate(item_ids)}

    rows = pair_df["user_id"].map(user_to_idx).to_numpy()
    cols = pair_df["edition_id"].map(item_to_idx).to_numpy()
    vals = pair_df[value_col].to_numpy(dtype=np.float32)

    matrix = sparse.csr_matrix(
        (vals, (rows, cols)),
        shape=(len(user_ids), len(item_ids)),
        dtype=np.float32,
    )
    matrix.sum_duplicates()
    return matrix, user_to_idx, item_ids


def _safe_normalize_rows(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    denom = np.linalg.norm(x, axis=1, keepdims=True)
    denom = np.where(denom < eps, 1.0, denom)
    return x / denom


class FallbackImplicitALS:
    def __init__(
        self,
        factors: int = 48,
        regularization: float = 0.06,
        iterations: int = 8,
        random_state: int = 42,
    ) -> None:
        self.factors = int(factors)
        self.regularization = float(regularization)
        self.iterations = int(iterations)
        self.random_state = int(random_state)
        self.user_factors: np.ndarray | None = None
        self.item_factors: np.ndarray | None = None

    def _least_squares(self, Cui: sparse.csr_matrix, Y: np.ndarray) -> np.ndarray:
        n_rows = Cui.shape[0]
        factors = Y.shape[1]
        X = np.zeros((n_rows, factors), dtype=np.float32)
        YtY = Y.T @ Y
        eye = np.eye(factors, dtype=np.float32)

        for u in tqdm(range(n_rows), desc="ALS solve", leave=False):
            start, end = Cui.indptr[u], Cui.indptr[u + 1]
            idx = Cui.indices[start:end]
            conf = Cui.data[start:end]
            if len(idx) == 0:
                continue
            A = YtY + self.regularization * eye
            b = np.zeros(factors, dtype=np.float32)
            for i, c in zip(idx, conf):
                y = Y[i]
                A += (c - 1.0) * np.outer(y, y)
                b += c * y
            X[u] = np.linalg.solve(A, b)
        return X

    def fit(self, Cui: sparse.csr_matrix) -> "FallbackImplicitALS":
        rng = np.random.default_rng(self.random_state)
        n_users, n_items = Cui.shape
        self.user_factors = (0.01 * rng.standard_normal((n_users, self.factors))).astype(np.float32)
        self.item_factors = (0.01 * rng.standard_normal((n_items, self.factors))).astype(np.float32)

        Ciu = Cui.T.tocsr()
        for _ in range(self.iterations):
            self.user_factors = self._least_squares(Cui, self.item_factors)
            self.item_factors = self._least_squares(Ciu, self.user_factors)
        return self


class ALSModel:
    def __init__(
        self,
        factors: int,
        regularization: float,
        iterations: int,
        random_state: int,
        use_gpu: bool,
    ) -> None:
        self.factors = factors
        self.regularization = regularization
        self.iterations = iterations
        self.random_state = random_state
        self.use_gpu = use_gpu

        self.backend_name: str = ""
        self.model: Any = None

        self.user_to_idx: dict[int, int] = {}
        self.idx_to_item: np.ndarray | None = None
        self.item_to_idx: dict[int, int] = {}
        self.seen_matrix: sparse.csr_matrix | None = None

        self.user_factors: np.ndarray | None = None
        self.item_factors: np.ndarray | None = None
        self.item_factors_norm: np.ndarray | None = None

    def fit(self, pair_df: pd.DataFrame, value_col: str) -> "ALSModel":
        matrix, user_to_idx, idx_to_item = build_interaction_matrix(pair_df, value_col=value_col)
        self.user_to_idx = user_to_idx
        self.idx_to_item = idx_to_item
        self.item_to_idx = {int(item): idx for idx, item in enumerate(idx_to_item)}
        self.seen_matrix = matrix.tocsr()

        if HAS_IMPLICIT:
            try:
                self.backend_name = "implicit"
                item_user = matrix.T.tocsr()
                try:
                    model = implicit.als.AlternatingLeastSquares(
                        factors=self.factors,
                        regularization=self.regularization,
                        iterations=self.iterations,
                        random_state=self.random_state,
                        use_gpu=bool(self.use_gpu),
                    )
                except TypeError:
                    model = implicit.als.AlternatingLeastSquares(
                        factors=self.factors,
                        regularization=self.regularization,
                        iterations=self.iterations,
                    )
                model.fit(item_user, show_progress=True)
                self.model = model
                self.user_factors = model.user_factors.astype(np.float32)
                self.item_factors = model.item_factors.astype(np.float32)
            except Exception as exc:
                print(f"implicit ALS failed, switch to fallback: {exc}")
                self.backend_name = "fallback"
                factors = min(self.factors, cfg.fallback_als_full_factors if self.factors >= 64 else cfg.fallback_als_recent_factors)
                iters = min(self.iterations, cfg.fallback_als_iterations)
                model = FallbackImplicitALS(
                    factors=factors,
                    regularization=self.regularization,
                    iterations=iters,
                    random_state=self.random_state,
                ).fit(matrix)
                self.model = model
                self.user_factors = model.user_factors
                self.item_factors = model.item_factors
        else:
            self.backend_name = "fallback"
            factors = min(self.factors, cfg.fallback_als_full_factors if self.factors >= 64 else cfg.fallback_als_recent_factors)
            iters = min(self.iterations, cfg.fallback_als_iterations)
            model = FallbackImplicitALS(
                factors=factors,
                regularization=self.regularization,
                iterations=iters,
                random_state=self.random_state,
            ).fit(matrix)
            self.model = model
            self.user_factors = model.user_factors
            self.item_factors = model.item_factors

        self.item_factors_norm = _safe_normalize_rows(self.item_factors)
        return self

    @property
    def n_users(self) -> int:
        return 0 if self.user_factors is None else int(self.user_factors.shape[0])

    @property
    def n_items(self) -> int:
        return 0 if self.item_factors is None else int(self.item_factors.shape[0])

    def has_user(self, user_id: int) -> bool:
        return int(user_id) in self.user_to_idx

    def has_item(self, item_id: int) -> bool:
        return int(item_id) in self.item_to_idx

    def score_pairs(self, user_ids: np.ndarray, item_ids: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        score = np.zeros(len(user_ids), dtype=np.float32)
        cos = np.zeros(len(user_ids), dtype=np.float32)

        if self.user_factors is None or self.item_factors is None:
            return score, cos

        valid_mask = np.array(
            [int(u) in self.user_to_idx and int(i) in self.item_to_idx for u, i in zip(user_ids, item_ids)],
            dtype=bool,
        )
        if valid_mask.any():
            u_idx = np.array([self.user_to_idx[int(u)] for u in user_ids[valid_mask]], dtype=np.int64)
            i_idx = np.array([self.item_to_idx[int(i)] for i in item_ids[valid_mask]], dtype=np.int64)
            u_vec = self.user_factors[u_idx]
            i_vec = self.item_factors[i_idx]
            score[valid_mask] = np.sum(u_vec * i_vec, axis=1).astype(np.float32)

            u_norm = np.linalg.norm(u_vec, axis=1)
            i_norm = np.linalg.norm(i_vec, axis=1)
            denom = np.maximum(u_norm * i_norm, 1e-12)
            cos[valid_mask] = (score[valid_mask] / denom).astype(np.float32)

        return score, cos

    def recommend_batch(
        self,
        user_ids: np.ndarray,
        n: int,
        filter_seen: bool = True,
        batch_size: int = 128,
    ) -> pd.DataFrame:
        if self.user_factors is None or self.item_factors is None or self.seen_matrix is None:
            return pd.DataFrame(columns=["user_id", "edition_id", "rank", "score"])

        rows: list[pd.DataFrame] = []
        item_ids = self.idx_to_item
        item_count = len(item_ids)

        valid_users = [int(u) for u in user_ids if int(u) in self.user_to_idx]
        if not valid_users:
            return pd.DataFrame(columns=["user_id", "edition_id", "rank", "score"])

        for start in tqdm(range(0, len(valid_users), batch_size), desc="ALS recommend", leave=False):
            batch_users = valid_users[start:start + batch_size]
            batch_idx = np.array([self.user_to_idx[u] for u in batch_users], dtype=np.int64)
            user_vecs = self.user_factors[batch_idx]
            scores = user_vecs @ self.item_factors.T

            for row_idx, user_id in enumerate(batch_users):
                if filter_seen:
                    seen_idx = self.seen_matrix[batch_idx[row_idx]].indices
                    scores[row_idx, seen_idx] = -1e18

            k = min(n, item_count)
            top_idx = np.argpartition(-scores, kth=max(k - 1, 0), axis=1)[:, :k]

            for row_idx, user_id in enumerate(batch_users):
                idx = top_idx[row_idx]
                sc = scores[row_idx, idx]
                order = np.argsort(-sc)
                idx = idx[order]
                sc = sc[order]
                rows.append(
                    pd.DataFrame(
                        {
                            "user_id": user_id,
                            "edition_id": item_ids[idx].astype(np.int64),
                            "rank": np.arange(1, len(idx) + 1, dtype=np.int32),
                            "score": sc.astype(np.float32),
                        }
                    )
                )

        return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["user_id", "edition_id", "rank", "score"])


def fit_als_models(pair_df: pd.DataFrame, cfg: CFG) -> dict[str, ALSModel]:
    full_model = ALSModel(
        factors=cfg.als_full_factors,
        regularization=cfg.als_regularization,
        iterations=cfg.als_iterations,
        random_state=cfg.seed,
        use_gpu=cfg.use_gpu,
    ).fit(pair_df, value_col="ui_weight")

    # recent ALS обучаем на той же pair таблице, но на recent weights
    recent_model = ALSModel(
        factors=cfg.als_recent_factors,
        regularization=cfg.als_regularization,
        iterations=cfg.als_recent_iterations,
        random_state=cfg.seed + 7,
        use_gpu=cfg.use_gpu,
    ).fit(pair_df, value_col="ui_recent_weight")

    print("ALS backends:", full_model.backend_name, recent_model.backend_name)
    return {"als_full": full_model, "als_recent": recent_model}


In [ ]:
def fit_knn_indexes(pair_df: pd.DataFrame, als_full: ALSModel) -> dict[str, Any]:
    # Latent KNN over normalized ALS item factors
    latent_nn = None
    if als_full.item_factors_norm is not None and len(als_full.item_factors_norm) > 1:
        latent_nn = NearestNeighbors(metric="cosine", algorithm="auto")
        latent_nn.fit(als_full.item_factors_norm)

    # Sparse KNN over item-user matrix
    matrix, _, idx_to_item = build_interaction_matrix(pair_df, value_col="ui_recent_weight")
    item_user = matrix.T.tocsr()
    item_user = normalize(item_user, norm="l2", axis=1)

    sparse_nn = None
    if item_user.shape[0] > 1:
        sparse_nn = NearestNeighbors(metric="cosine", algorithm="brute")
        sparse_nn.fit(item_user)

    sparse_item_to_idx = {int(item): idx for idx, item in enumerate(idx_to_item)}

    # Read-only sparse KNN (captures read-read graph independent from wishlist)
    read_df = pair_df[pair_df["ui_reads"] > 0][["user_id", "edition_id", "ui_reads"]].copy()
    read_sparse_nn = None
    read_item_user = sparse.csr_matrix((0, 0), dtype=np.float32)
    read_idx_to_item = np.array([], dtype=np.int64)
    read_item_to_idx: dict[int, int] = {}
    if not read_df.empty:
        read_matrix, _, read_idx_to_item = build_interaction_matrix(read_df, value_col="ui_reads")
        read_item_user = normalize(read_matrix.T.tocsr(), norm="l2", axis=1)
        if read_item_user.shape[0] > 1:
            read_sparse_nn = NearestNeighbors(metric="cosine", algorithm="brute")
            read_sparse_nn.fit(read_item_user)
        read_item_to_idx = {int(item): idx for idx, item in enumerate(read_idx_to_item)}
    return {
        "latent_nn": latent_nn,
        "sparse_nn": sparse_nn,
        "sparse_item_user": item_user,
        "sparse_idx_to_item": idx_to_item,
        "sparse_item_to_idx": sparse_item_to_idx,
        "read_sparse_nn": read_sparse_nn,
        "read_sparse_item_user": read_item_user,
        "read_sparse_idx_to_item": read_idx_to_item,
        "read_sparse_item_to_idx": read_item_to_idx,
    }


def query_neighbors_dense(
    model: ALSModel,
    seed_items: np.ndarray,
    nn_model: NearestNeighbors | None,
    topk: int,
) -> pd.DataFrame:
    if nn_model is None or model.item_factors_norm is None:
        return pd.DataFrame(columns=["seed_item_id", "edition_id", "nbr_rank", "sim"])

    uniq_items = np.unique(seed_items.astype(np.int64))
    valid = [it for it in uniq_items if int(it) in model.item_to_idx]
    if not valid:
        return pd.DataFrame(columns=["seed_item_id", "edition_id", "nbr_rank", "sim"])

    idx = np.array([model.item_to_idx[int(it)] for it in valid], dtype=np.int64)
    k = min(topk + 1, model.n_items)
    dists, inds = nn_model.kneighbors(model.item_factors_norm[idx], n_neighbors=k)

    rows = []
    for seed_item, dist_row, ind_row in zip(valid, dists, inds):
        sim = 1.0 - dist_row
        nbr_items = model.idx_to_item[ind_row].astype(np.int64)
        df = pd.DataFrame(
            {
                "seed_item_id": int(seed_item),
                "edition_id": nbr_items,
                "sim": sim.astype(np.float32),
            }
        )
        df = df[df["edition_id"] != int(seed_item)].head(topk).copy()
        df["nbr_rank"] = np.arange(1, len(df) + 1, dtype=np.int32)
        rows.append(df)

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["seed_item_id", "edition_id", "nbr_rank", "sim"])


def query_neighbors_sparse(
    seed_items: np.ndarray,
    sparse_item_to_idx: dict[int, int],
    sparse_idx_to_item: np.ndarray,
    sparse_item_user: sparse.csr_matrix,
    nn_model: NearestNeighbors | None,
    topk: int,
) -> pd.DataFrame:
    if nn_model is None:
        return pd.DataFrame(columns=["seed_item_id", "edition_id", "nbr_rank", "sim"])

    uniq_items = np.unique(seed_items.astype(np.int64))
    valid = [it for it in uniq_items if int(it) in sparse_item_to_idx]
    if not valid:
        return pd.DataFrame(columns=["seed_item_id", "edition_id", "nbr_rank", "sim"])

    idx = np.array([sparse_item_to_idx[int(it)] for it in valid], dtype=np.int64)
    k = min(topk + 1, sparse_item_user.shape[0])
    dists, inds = nn_model.kneighbors(sparse_item_user[idx], n_neighbors=k)

    rows = []
    for seed_item, dist_row, ind_row in zip(valid, dists, inds):
        sim = 1.0 - dist_row
        nbr_items = sparse_idx_to_item[ind_row].astype(np.int64)
        df = pd.DataFrame(
            {
                "seed_item_id": int(seed_item),
                "edition_id": nbr_items,
                "sim": sim.astype(np.float32),
            }
        )
        df = df[df["edition_id"] != int(seed_item)].head(topk).copy()
        df["nbr_rank"] = np.arange(1, len(df) + 1, dtype=np.int32)
        rows.append(df)

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["seed_item_id", "edition_id", "nbr_rank", "sim"])


def build_context(
    observed: pd.DataFrame,
    ref_ts: pd.Timestamp,
    edition_meta: pd.DataFrame,
    edition_genres: pd.DataFrame,
    users: pd.DataFrame,
    cfg: CFG,
) -> dict[str, Any]:
    observed_w = add_interaction_weights(observed, ref_ts, cfg)
    pair_df = build_pair_aggregates(observed, ref_ts, cfg)
    user_features = build_user_features(observed_w, pair_df, users, ref_ts)
    item_features = build_item_features(observed_w, pair_df, edition_meta, ref_ts)
    affinity = build_affinity_tables(pair_df, edition_meta, edition_genres)
    seeds = build_seed_items(pair_df, cfg)
    pop = build_popularity_tables(pair_df, affinity, users, cfg)
    als = fit_als_models(pair_df, cfg)
    knn = fit_knn_indexes(pair_df, als["als_full"])

    seen_pairs = pair_df[["user_id", "edition_id"]].drop_duplicates()

    return {
        "ref_ts": ref_ts,
        "observed_w": observed_w,
        "pair_df": pair_df,
        "user_features": user_features,
        "item_features": item_features,
        "affinity": affinity,
        "seeds": seeds,
        "pop": pop,
        "als": als,
        "knn": knn,
        "seen_pairs": seen_pairs,
        "edition_meta": edition_meta,
        "users": users,
    }


In [ ]:
def rank_to_score(series: pd.Series, denom_shift: float = 1.0) -> pd.Series:
    return 1.0 / (series.astype(float) + denom_shift)


def build_als_candidates(
    context: dict[str, Any],
    user_ids: np.ndarray,
    model_key: str,
    topk: int,
    prefix: str,
) -> pd.DataFrame:
    model: ALSModel = context["als"][model_key]
    df = model.recommend_batch(user_ids=user_ids, n=topk, filter_seen=True, batch_size=cfg.batch_size_reco)
    if df.empty:
        return pd.DataFrame(columns=["user_id", "edition_id", f"{prefix}_rank", f"{prefix}_score"])
    df = df.rename(columns={"rank": f"{prefix}_rank", "score": f"{prefix}_score"})
    return df


def build_knn_candidates_from_seeds(
    context: dict[str, Any],
    user_ids: np.ndarray,
    topk: int,
    prefix: str,
    sparse_mode: bool = False,
    read_mode: bool = False,
) -> pd.DataFrame:
    seeds = context["seeds"]
    seed_sub = seeds[seeds["user_id"].isin(user_ids)].copy()
    if seed_sub.empty:
        cols = ["user_id", "edition_id", f"{prefix}_score", f"{prefix}_max_sim", f"{prefix}_count", f"{prefix}_rank", f"{prefix}_best_seed_rank"]
        return pd.DataFrame(columns=cols)

    if read_mode:
        nbr = query_neighbors_sparse(
            seed_items=seed_sub["edition_id"].to_numpy(),
            sparse_item_to_idx=context["knn"]["read_sparse_item_to_idx"],
            sparse_idx_to_item=context["knn"]["read_sparse_idx_to_item"],
            sparse_item_user=context["knn"]["read_sparse_item_user"],
            nn_model=context["knn"]["read_sparse_nn"],
            topk=topk,
        )
    elif sparse_mode:
        nbr = query_neighbors_sparse(
            seed_items=seed_sub["edition_id"].to_numpy(),
            sparse_item_to_idx=context["knn"]["sparse_item_to_idx"],
            sparse_idx_to_item=context["knn"]["sparse_idx_to_item"],
            sparse_item_user=context["knn"]["sparse_item_user"],
            nn_model=context["knn"]["sparse_nn"],
            topk=topk,
        )
    else:
        nbr = query_neighbors_dense(
            model=context["als"]["als_full"],
            seed_items=seed_sub["edition_id"].to_numpy(),
            nn_model=context["knn"]["latent_nn"],
            topk=topk,
        )

    if nbr.empty:
        cols = ["user_id", "edition_id", f"{prefix}_score", f"{prefix}_max_sim", f"{prefix}_count", f"{prefix}_rank", f"{prefix}_best_seed_rank"]
        return pd.DataFrame(columns=cols)

    merged = seed_sub[["user_id", "edition_id", "seed_weight", "seed_rank"]].rename(columns={"edition_id": "seed_item_id"}).merge(
        nbr, on="seed_item_id", how="inner"
    )
    merged[f"{prefix}_weighted_sim"] = (merged["seed_weight"] * merged["sim"]).astype("float32")

    agg = (
        merged.groupby(["user_id", "edition_id"], as_index=False)
        .agg(
            **{
                f"{prefix}_score": (f"{prefix}_weighted_sim", "sum"),
                f"{prefix}_max_sim": ("sim", "max"),
                f"{prefix}_count": ("sim", "size"),
                f"{prefix}_rank": ("nbr_rank", "min"),
                f"{prefix}_best_seed_rank": ("seed_rank", "min"),
            }
        )
    )
    return agg


def build_entity_pop_candidates(
    user_top: pd.DataFrame,
    entity_item_pop: pd.DataFrame,
    user_ids: np.ndarray,
    entity_col: str,
    user_weight_col: str,
    item_score_col: str,
    item_rank_col: str,
    prefix: str,
) -> pd.DataFrame:
    user_sub = user_top[user_top["user_id"].isin(user_ids)].copy()
    if user_sub.empty:
        cols = ["user_id", "edition_id", f"{prefix}_score", f"{prefix}_rank", f"{prefix}_entity_rank"]
        return pd.DataFrame(columns=cols)

    merged = user_sub.merge(entity_item_pop, on=entity_col, how="inner")
    merged[f"{prefix}_score"] = (merged[user_weight_col] * merged[item_score_col]).astype("float32")

    agg = (
        merged.groupby(["user_id", "edition_id"], as_index=False)
        .agg(
            **{
                f"{prefix}_score": (f"{prefix}_score", "max"),
                f"{prefix}_rank": (item_rank_col, "min"),
                f"{prefix}_entity_rank": (user_weight_col, "max"),
            }
        )
    )
    return agg


def build_demo_candidates(
    context: dict[str, Any],
    user_ids: np.ndarray,
) -> pd.DataFrame:
    user_demo = context["users"][["user_id", "gender", "age_bucket"]]
    user_sub = user_demo[user_demo["user_id"].isin(user_ids)].copy()
    if user_sub.empty:
        return pd.DataFrame(columns=["user_id", "edition_id", "demo_score", "demo_rank", "gender_score", "gender_rank"])

    demo = user_sub.merge(context["pop"]["demo_pop"], on=["gender", "age_bucket"], how="left")
    gender = user_sub.merge(context["pop"]["gender_pop"], on=["gender"], how="left")

    demo = demo[demo["edition_id"].notna()].copy()
    gender = gender[gender["edition_id"].notna()].copy()

    if not demo.empty:
        demo["edition_id"] = demo["edition_id"].astype("int64")
    if not gender.empty:
        gender["edition_id"] = gender["edition_id"].astype("int64")

    out = demo[["user_id", "edition_id", "demo_score", "demo_rank"]].merge(
        gender[["user_id", "edition_id", "gender_score", "gender_rank"]],
        on=["user_id", "edition_id"],
        how="outer",
    )
    return out


def build_global_candidates(
    context: dict[str, Any],
    user_ids: np.ndarray,
) -> pd.DataFrame:
    global_pop = context["pop"]["global_pop"][["edition_id", "pop_score", "pop_rank"]].copy()
    if global_pop.empty or len(user_ids) == 0:
        return pd.DataFrame(columns=["user_id", "edition_id", "pop_score", "pop_rank"])

    users_df = pd.DataFrame({"user_id": user_ids.astype(np.int64)})
    users_df["_tmp"] = 1
    global_pop["_tmp"] = 1
    out = users_df.merge(global_pop, on="_tmp", how="inner").drop(columns="_tmp")
    return out


def merge_candidate_sources(
    source_frames: dict[str, pd.DataFrame],
    seen_pairs: pd.DataFrame,
    max_candidates_per_user: int,
) -> pd.DataFrame:
    parts = []
    for _, df in source_frames.items():
        if df is not None and not df.empty:
            parts.append(df[["user_id", "edition_id"]].copy())

    if not parts:
        return pd.DataFrame(columns=["user_id", "edition_id", "pre_score"])

    pool = pd.concat(parts, ignore_index=True).drop_duplicates()
    for _, df in source_frames.items():
        if df is None or df.empty:
            continue
        extra_cols = [c for c in df.columns if c not in {"user_id", "edition_id"}]
        pool = pool.merge(df[["user_id", "edition_id"] + extra_cols], on=["user_id", "edition_id"], how="left")

    pool = pool.merge(seen_pairs.assign(_seen=1), on=["user_id", "edition_id"], how="left")
    pool = pool[pool["_seen"].isna()].drop(columns="_seen")

    score = np.zeros(len(pool), dtype=np.float32)

    if "als_full_rank" in pool:
        score += 2.8 * rank_to_score(pool["als_full_rank"].fillna(1e9)).to_numpy(dtype=np.float32)
    if "als_recent_rank" in pool:
        score += 2.2 * rank_to_score(pool["als_recent_rank"].fillna(1e9)).to_numpy(dtype=np.float32)
    if "latent_knn_score" in pool:
        score += 1.6 * pool["latent_knn_score"].fillna(0).to_numpy(dtype=np.float32)
    if "sparse_knn_score" in pool:
        score += 1.25 * pool["sparse_knn_score"].fillna(0).to_numpy(dtype=np.float32)
    if "read_knn_score" in pool:
        score += 1.55 * pool["read_knn_score"].fillna(0).to_numpy(dtype=np.float32)
    if "author_src_rank" in pool:
        score += 1.1 * rank_to_score(pool["author_src_rank"].fillna(1e9)).to_numpy(dtype=np.float32)
    if "book_src_rank" in pool:
        score += 0.9 * rank_to_score(pool["book_src_rank"].fillna(1e9)).to_numpy(dtype=np.float32)
    if "genre_src_rank" in pool:
        score += 0.8 * rank_to_score(pool["genre_src_rank"].fillna(1e9)).to_numpy(dtype=np.float32)
    if "demo_rank" in pool:
        score += 0.55 * rank_to_score(pool["demo_rank"].fillna(1e9)).to_numpy(dtype=np.float32)
    if "gender_rank" in pool:
        score += 0.35 * rank_to_score(pool["gender_rank"].fillna(1e9)).to_numpy(dtype=np.float32)
    if "pop_rank" in pool:
        score += 0.45 * rank_to_score(pool["pop_rank"].fillna(1e9)).to_numpy(dtype=np.float32)

    rank_cols = [c for c in pool.columns if c.endswith("_rank")]
    if rank_cols:
        pool["source_count"] = pool[rank_cols].notna().sum(axis=1).astype("int16")
        score += 0.12 * pool["source_count"].to_numpy(dtype=np.float32)
    else:
        pool["source_count"] = 0

    pool["pre_score"] = score.astype(np.float32)
    pool = (
        pool.sort_values(["user_id", "pre_score"], ascending=[True, False])
        .groupby("user_id")
        .head(max_candidates_per_user)
        .reset_index(drop=True)
    )
    return pool


def build_candidate_pool(context: dict[str, Any], user_ids: np.ndarray, cfg: CFG) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    sources: dict[str, pd.DataFrame] = {}

    sources["als_full"] = build_als_candidates(context, user_ids, model_key="als_full", topk=cfg.als_full_topk, prefix="als_full")
    sources["als_recent"] = build_als_candidates(context, user_ids, model_key="als_recent", topk=cfg.als_recent_topk, prefix="als_recent")

    sources["latent_knn"] = build_knn_candidates_from_seeds(
        context, user_ids=user_ids, topk=cfg.latent_knn_neighbors, prefix="latent_knn", sparse_mode=False
    )
    sources["sparse_knn"] = build_knn_candidates_from_seeds(
        context, user_ids=user_ids, topk=cfg.sparse_knn_neighbors, prefix="sparse_knn", sparse_mode=True
    )
    sources["read_knn"] = build_knn_candidates_from_seeds(
        context, user_ids=user_ids, topk=cfg.read_knn_neighbors, prefix="read_knn", read_mode=True
    )

    sources["author_src"] = build_entity_pop_candidates(
        user_top=context["pop"]["user_author_top"],
        entity_item_pop=context["pop"]["author_item_pop"],
        user_ids=user_ids,
        entity_col="author_id",
        user_weight_col="ua_recent_weight",
        item_score_col="author_item_score",
        item_rank_col="author_item_rank",
        prefix="author_src",
    )
    sources["book_src"] = build_entity_pop_candidates(
        user_top=context["pop"]["user_book_top"],
        entity_item_pop=context["pop"]["book_item_pop"],
        user_ids=user_ids,
        entity_col="book_id",
        user_weight_col="ub_recent_weight",
        item_score_col="book_item_score",
        item_rank_col="book_item_rank",
        prefix="book_src",
    )
    sources["genre_src"] = build_entity_pop_candidates(
        user_top=context["pop"]["user_genre_top"],
        entity_item_pop=context["pop"]["genre_item_pop"],
        user_ids=user_ids,
        entity_col="genre_id",
        user_weight_col="ug_recent_weight",
        item_score_col="genre_item_score",
        item_rank_col="genre_item_rank",
        prefix="genre_src",
    )

    sources["demo_src"] = build_demo_candidates(context, user_ids)
    sources["global_src"] = build_global_candidates(context, user_ids)

    pool = merge_candidate_sources(
        source_frames=sources,
        seen_pairs=context["seen_pairs"],
        max_candidates_per_user=cfg.max_candidates_per_user,
    )
    return pool, sources


In [ ]:
def add_als_pair_features(df: pd.DataFrame, model: ALSModel, prefix: str) -> pd.DataFrame:
    user_ids = df["user_id"].to_numpy(dtype=np.int64)
    item_ids = df["edition_id"].to_numpy(dtype=np.int64)
    score, cos = model.score_pairs(user_ids, item_ids)
    df[f"{prefix}_pair_score"] = score.astype(np.float32)
    df[f"{prefix}_pair_cos"] = cos.astype(np.float32)

    if model.user_factors is not None and model.item_factors is not None:
        user_norm = np.zeros(len(df), dtype=np.float32)
        item_norm = np.zeros(len(df), dtype=np.float32)
        valid_mask = np.array(
            [int(u) in model.user_to_idx and int(i) in model.item_to_idx for u, i in zip(user_ids, item_ids)],
            dtype=bool,
        )
        if valid_mask.any():
            u_idx = np.array([model.user_to_idx[int(u)] for u in user_ids[valid_mask]], dtype=np.int64)
            i_idx = np.array([model.item_to_idx[int(i)] for i in item_ids[valid_mask]], dtype=np.int64)
            user_norm[valid_mask] = np.linalg.norm(model.user_factors[u_idx], axis=1).astype(np.float32)
            item_norm[valid_mask] = np.linalg.norm(model.item_factors[i_idx], axis=1).astype(np.float32)
        df[f"{prefix}_user_norm"] = user_norm
        df[f"{prefix}_item_norm"] = item_norm
        df[f"{prefix}_norm_gap"] = np.abs(user_norm - item_norm).astype(np.float32)
    else:
        df[f"{prefix}_user_norm"] = 0.0
        df[f"{prefix}_item_norm"] = 0.0
        df[f"{prefix}_norm_gap"] = 0.0
    return df


def choose_top_source(df: pd.DataFrame) -> pd.Series:
    source_candidates: dict[str, pd.Series] = {}

    if "als_full_rank" in df:
        source_candidates["als_full"] = 2.8 * rank_to_score(df["als_full_rank"].fillna(1e9))
    if "als_recent_rank" in df:
        source_candidates["als_recent"] = 2.2 * rank_to_score(df["als_recent_rank"].fillna(1e9))
    if "latent_knn_score" in df:
        source_candidates["latent_knn"] = 1.6 * df["latent_knn_score"].fillna(0)
    if "sparse_knn_score" in df:
        source_candidates["sparse_knn"] = 1.25 * df["sparse_knn_score"].fillna(0)
    if "read_knn_score" in df:
        source_candidates["read_knn"] = 1.55 * df["read_knn_score"].fillna(0)
    if "author_src_rank" in df:
        source_candidates["author_src"] = 1.1 * rank_to_score(df["author_src_rank"].fillna(1e9))
    if "book_src_rank" in df:
        source_candidates["book_src"] = 0.9 * rank_to_score(df["book_src_rank"].fillna(1e9))
    if "genre_src_rank" in df:
        source_candidates["genre_src"] = 0.8 * rank_to_score(df["genre_src_rank"].fillna(1e9))
    if "demo_rank" in df:
        source_candidates["demo"] = 0.55 * rank_to_score(df["demo_rank"].fillna(1e9))
    if "pop_rank" in df:
        source_candidates["global"] = 0.45 * rank_to_score(df["pop_rank"].fillna(1e9))

    if not source_candidates:
        return pd.Series(["none"] * len(df), index=df.index)

    source_scores = pd.DataFrame(source_candidates)
    return source_scores.idxmax(axis=1)


def assemble_candidate_features(
    context: dict[str, Any],
    candidate_pool: pd.DataFrame,
    hidden_pairs: pd.DataFrame | None = None,
) -> pd.DataFrame:
    df = candidate_pool.copy()

    # User/item tables
    df = df.merge(context["item_features"], on="edition_id", how="left")
    df = df.merge(context["user_features"], on="user_id", how="left")

    # Affinity merges
    aff = context["affinity"]
    df = df.merge(
        aff["user_author"][["user_id", "author_id", "ua_weight", "ua_recent_weight", "ua_items", "ua_last_days", "ua_rank"]],
        on=["user_id", "author_id"],
        how="left",
    )
    df = df.merge(
        aff["user_book"][["user_id", "book_id", "ub_weight", "ub_recent_weight", "ub_items", "ub_last_days", "ub_rank"]],
        on=["user_id", "book_id"],
        how="left",
    )
    df = df.merge(
        aff["user_lang"][["user_id", "language_id", "ul_weight", "ul_recent_weight", "ul_items", "ul_last_days", "ul_rank"]],
        on=["user_id", "language_id"],
        how="left",
    )
    df = df.merge(
        aff["user_pub"][["user_id", "publisher_id", "up_weight", "up_recent_weight", "up_items", "up_last_days", "up_rank"]],
        on=["user_id", "publisher_id"],
        how="left",
    )

    # ALS pair features
    df = add_als_pair_features(df, context["als"]["als_full"], prefix="als_full")
    df = add_als_pair_features(df, context["als"]["als_recent"], prefix="als_recent")

    # Cross features
    df["user_item_pop_ratio"] = (df["item_recent_weight"].fillna(0) / np.maximum(df["user_recent_weight"].fillna(0), 1e-6)).astype("float32")
    df["user_item_weight_ratio"] = (df["item_weight"].fillna(0) / np.maximum(df["user_weight"].fillna(0), 1e-6)).astype("float32")
    df["age_restriction_gap"] = (df["age"].fillna(-1) - df["age_restriction"].fillna(-1)).astype("float32")
    df["pub_year_gap"] = (2025 - df["publication_year"].fillna(2025)).astype("float32")
    df["is_adult_item_for_minor"] = ((df["age_restriction"].fillna(-1) >= 18) & (df["age"].fillna(100) < 18)).astype("int8")
    df["is_cold_user"] = (df["user_seen_items"].fillna(0) == 0).astype("int8")
    df["is_tail_item"] = (df["item_users"].fillna(0) <= 2).astype("int8")
    user_hot = df["user_hot_ratio"] if "user_hot_ratio" in df.columns else 0.0
    item_hot = df["item_hot_ratio"] if "item_hot_ratio" in df.columns else 0.0
    user_recent_ratio = df["user_recent_to_all_ratio"] if "user_recent_to_all_ratio" in df.columns else 0.0
    item_trend = df["item_trend_ratio"] if "item_trend_ratio" in df.columns else 0.0
    df["cross_hot"] = (user_hot * item_hot).astype("float32")
    df["cross_trend_user_item"] = (user_recent_ratio * item_trend).astype("float32")
    df["source_count"] = df["source_count"].fillna(0).astype("int16")
    df["top_source"] = choose_top_source(df).astype(str)

    # Fill numeric missing values
    numeric_cols = df.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    for col in numeric_cols:
        if str(df[col].dtype).startswith("float"):
            df[col] = df[col].fillna(0).astype("float32")
        elif str(df[col].dtype).startswith("int"):
            df[col] = df[col].fillna(0)
        else:
            df[col] = df[col].fillna(0)

    # Categorical columns for CatBoost
    cat_cols = ["author_id", "book_id", "language_id", "publisher_id", "gender", "age_bucket", "top_source"]
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].fillna(-1).astype(str)

    if hidden_pairs is not None:
        df = df.merge(hidden_pairs.assign(label=1), on=["user_id", "edition_id"], how="left")
        df["label"] = df["label"].fillna(0).astype("int8")

    return df


def inject_missed_positives(candidate_pool: pd.DataFrame, hidden_pairs: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    missing = hidden_pairs.merge(candidate_pool[["user_id", "edition_id"]], on=["user_id", "edition_id"], how="left", indicator=True)
    missing = missing[missing["_merge"] == "left_only"][["user_id", "edition_id"]].copy()
    if missing.empty:
        return candidate_pool, 0
    missing["pre_score"] = 0.0
    missing["forced_positive"] = 1
    out = pd.concat([candidate_pool, missing], ignore_index=True, sort=False).drop_duplicates(["user_id", "edition_id"])
    return out, int(len(missing))


def downsample_negatives_per_user(
    df: pd.DataFrame,
    max_negatives_per_user: int,
    seed: int,
) -> pd.DataFrame:
    if "label" not in df.columns:
        return df

    rng = np.random.default_rng(seed)
    pos = df[df["label"] == 1].copy()
    neg = df[df["label"] == 0].copy()

    keep_parts = [pos]
    if neg.empty:
        return pos.copy()

    neg = neg.sort_values(["user_id", "pre_score"], ascending=[True, False]).copy()
    hard = neg.groupby("user_id").head(max_negatives_per_user // 2)

    rest = neg.merge(hard[["user_id", "edition_id"]].assign(_hard=1), on=["user_id", "edition_id"], how="left")
    rest = rest[rest["_hard"].isna()].drop(columns="_hard")

    sampled_rest = []
    n_random = max_negatives_per_user - max_negatives_per_user // 2
    for user_id, grp in rest.groupby("user_id", sort=False):
        if len(grp) <= n_random:
            sampled_rest.append(grp)
        else:
            idx = rng.choice(grp.index.to_numpy(), size=n_random, replace=False)
            sampled_rest.append(grp.loc[idx])

    if sampled_rest:
        keep_parts.append(hard)
        keep_parts.append(pd.concat(sampled_rest, ignore_index=True))
    else:
        keep_parts.append(hard)

    out = pd.concat(keep_parts, ignore_index=True)
    out = out.sort_values(["user_id", "pre_score"], ascending=[True, False]).reset_index(drop=True)
    return out


In [ ]:
def prepare_fold_dataset(
    interactions: pd.DataFrame,
    fold: FoldSpec,
    edition_meta: pd.DataFrame,
    edition_genres: pd.DataFrame,
    users: pd.DataFrame,
    cfg: CFG,
) -> dict[str, Any]:
    observed, hidden_pairs, fold_users = build_masked_fold(interactions, fold, cfg)
    context = build_context(
        observed=observed,
        ref_ts=fold.window_end,
        edition_meta=edition_meta,
        edition_genres=edition_genres,
        users=users,
        cfg=cfg,
    )

    candidate_pool, source_frames = build_candidate_pool(context, fold_users, cfg)
    recall_50 = candidate_recall_at_k(candidate_pool, hidden_pairs, k=50)
    recall_100 = candidate_recall_at_k(candidate_pool, hidden_pairs, k=100)
    recall_200 = candidate_recall_at_k(candidate_pool, hidden_pairs, k=200)
    print(f"[{fold.name}] candidate recall@50={recall_50:.5f} @100={recall_100:.5f} @200={recall_200:.5f}")

    candidate_pool, missed_cnt = inject_missed_positives(candidate_pool, hidden_pairs)
    if missed_cnt:
        print(f"[{fold.name}] injected missed positives: {missed_cnt}")

    feat_df = assemble_candidate_features(context, candidate_pool, hidden_pairs=hidden_pairs)
    base_ndcg = evaluate_scored_candidates(feat_df, hidden_pairs, score_col="pre_score", k=TOP_K)
    print(f"[{fold.name}] heuristic pre-score NDCG@20 = {base_ndcg:.5f}")

    # Не держим тяжелые context/observed в памяти после сборки признаков
    del context, observed, source_frames, candidate_pool
    gc.collect()

    return {
        "fold": fold,
        "hidden_pairs": hidden_pairs,
        "fold_users": fold_users,
        "features": feat_df,
        "heuristic_ndcg": base_ndcg,
    }


fold_artifacts: list[dict[str, Any]] = []
for fold in folds:
    print("=" * 100)
    art = prepare_fold_dataset(
        interactions=interactions,
        fold=fold,
        edition_meta=edition_meta,
        edition_genres=edition_genres,
        users=users,
        cfg=cfg,
    )
    if art["features"].empty or art["hidden_pairs"].empty:
        print(f"Skip {fold.name}: empty features or empty hidden pairs")
        continue
    fold_artifacts.append(art)
    gc.collect()


In [ ]:
from catboost import CatBoostRanker, Pool


if len(fold_artifacts) < 2:
    raise RuntimeError("Нужно минимум 2 pseudo folds: train + validation.")
valid_art = fold_artifacts[-1]
train_arts = fold_artifacts[:-1]
if len(train_arts) == 0:
    raise RuntimeError("Для обучения нужен хотя бы один training pseudo fold. Увеличьте cfg.n_pseudo_folds.")

train_df = pd.concat([art["features"] for art in train_arts], ignore_index=True)
valid_df = valid_art["features"].copy()

train_df = downsample_negatives_per_user(train_df, max_negatives_per_user=cfg.max_negatives_per_user, seed=cfg.seed)
valid_df = valid_df.sort_values(["user_id", "pre_score"], ascending=[True, False]).reset_index(drop=True)

# Уберём слишком маленькие группы из train
group_sizes = train_df.groupby("user_id")["edition_id"].size()
group_pos = train_df.groupby("user_id")["label"].sum()
good_users = set(group_sizes[group_sizes >= cfg.min_group_size_for_train].index.tolist())
good_users &= set(group_pos[group_pos > 0].index.tolist())
train_df = train_df[train_df["user_id"].isin(good_users)].copy()

print("train_df:", train_df.shape)
print("valid_df:", valid_df.shape)
print("train positives:", int(train_df["label"].sum()), " / ", len(train_df))
print("valid positives:", int(valid_df["label"].sum()), " / ", len(valid_df))

DROP_COLS = {"label", "fold_name"}
KEY_COLS = ["user_id", "edition_id"]
CAT_COLS = ["author_id", "book_id", "language_id", "publisher_id", "gender", "age_bucket", "top_source"]

feature_cols = [c for c in train_df.columns if c not in DROP_COLS and c not in []]
# user_id / edition_id оставляем вне фичей, но в train_df пригодятся для group / evaluation
feature_cols = [c for c in feature_cols if c not in KEY_COLS]

cat_cols = [c for c in CAT_COLS if c in feature_cols]
num_cols = [c for c in feature_cols if c not in cat_cols]

for c in num_cols:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce").fillna(0).astype("float32")
    valid_df[c] = pd.to_numeric(valid_df[c], errors="coerce").fillna(0).astype("float32")

for c in cat_cols:
    train_df[c] = train_df[c].fillna("UNK").astype(str)
    valid_df[c] = valid_df[c].fillna("UNK").astype(str)

train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df["label"],
    group_id=train_df["user_id"],
    cat_features=cat_cols,
    feature_names=feature_cols,
)
valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df["label"],
    group_id=valid_df["user_id"],
    cat_features=cat_cols,
    feature_names=feature_cols,
)

task_type = "GPU" if cfg.use_gpu else "CPU"


In [ ]:
loss_a = "YetiRankPairwise" if task_type == "GPU" else "YetiRank"
depth_a = 8 if task_type == "GPU" else 10

params_a = dict(
    loss_function=loss_a,
    eval_metric="NDCG:top=20",
    task_type=task_type,
    random_seed=cfg.seed,
    iterations=cfg.cb_iter_a,
    learning_rate=cfg.cb_lr_a,
    depth=depth_a,
    l2_leaf_reg=8.0,
    bootstrap_type="Bernoulli",
    subsample=0.8,
    verbose=100,
    od_type="Iter",
    od_wait=150,
    border_count=32 if task_type == "GPU" else 254,
)

params_b = dict(
    loss_function="QuerySoftMax",
    eval_metric="NDCG:top=20",
    task_type=task_type,
    random_seed=cfg.seed + 17,
    iterations=cfg.cb_iter_b,
    learning_rate=cfg.cb_lr_b,
    depth=10 if task_type == "CPU" else 10,
    l2_leaf_reg=6.0,
    bootstrap_type="Bernoulli",
    subsample=0.85,
    verbose=100,
    od_type="Iter",
    od_wait=150,
    border_count=128 if task_type == "GPU" else 254,
)

model_a = CatBoostRanker(**params_a)
model_b = CatBoostRanker(**params_b)

model_a.fit(train_pool, eval_set=valid_pool, use_best_model=True)
model_b.fit(train_pool, eval_set=valid_pool, use_best_model=True)

valid_pred_a = model_a.predict(valid_pool)
valid_pred_b = model_b.predict(valid_pool)

valid_eval = valid_df[["user_id", "edition_id"]].copy()
valid_eval["pred_a"] = valid_pred_a
valid_eval["pred_b"] = valid_pred_b
valid_eval["pre_score"] = valid_df["pre_score"].to_numpy()

score_pre = evaluate_scored_candidates(valid_eval.rename(columns={"pre_score": "score"}), valid_art["hidden_pairs"], score_col="score")
score_a = evaluate_scored_candidates(valid_eval.rename(columns={"pred_a": "score"}), valid_art["hidden_pairs"], score_col="score")
score_b = evaluate_scored_candidates(valid_eval.rename(columns={"pred_b": "score"}), valid_art["hidden_pairs"], score_col="score")

print("Validation NDCG@20")
print("heuristic:", round(score_pre, 6))
print("model_a  :", round(score_a, 6))
print("model_b  :", round(score_b, 6))


def find_best_blend_three(valid_eval: pd.DataFrame, hidden_pairs: pd.DataFrame) -> tuple[tuple[float, float, float], float]:
    best = (0.45, 0.45, 0.10)
    best_score = -1.0
    grid = np.linspace(0.0, 1.0, 11)
    for w_a in grid:
        for w_b in grid:
            w_h = 1.0 - w_a - w_b
            if w_h < 0.0:
                continue
            tmp = valid_eval[["user_id", "edition_id"]].copy()
            tmp["score"] = (
                w_a * valid_eval["pred_a"] + w_b * valid_eval["pred_b"] + w_h * valid_eval["pre_score"]
            ).astype(np.float32)
            score = evaluate_scored_candidates(tmp, hidden_pairs, score_col="score")
            if score > best_score:
                best_score = score
                best = (float(w_a), float(w_b), float(w_h))
    return best, best_score


blend_ws, blend_score = find_best_blend_three(valid_eval, valid_art["hidden_pairs"])
print("best blend weights (a, b, heuristic):", blend_ws)
print("best blend ndcg@20:", round(blend_score, 6))


In [ ]:
fi_a = pd.DataFrame(
    {"feature": feature_cols, "importance": model_a.get_feature_importance(train_pool)}
).sort_values("importance", ascending=False)
fi_b = pd.DataFrame(
    {"feature": feature_cols, "importance": model_b.get_feature_importance(train_pool)}
).sort_values("importance", ascending=False)

display(fi_a.head(40))
display(fi_b.head(40))


In [ ]:
all_train_df = pd.concat([art["features"] for art in fold_artifacts], ignore_index=True)
all_train_df = downsample_negatives_per_user(all_train_df, max_negatives_per_user=cfg.max_negatives_per_user, seed=cfg.seed + 123)

group_sizes = all_train_df.groupby("user_id")["edition_id"].size()
group_pos = all_train_df.groupby("user_id")["label"].sum()
good_users = set(group_sizes[group_sizes >= cfg.min_group_size_for_train].index.tolist())
good_users &= set(group_pos[group_pos > 0].index.tolist())
all_train_df = all_train_df[all_train_df["user_id"].isin(good_users)].copy()

for c in num_cols:
    all_train_df[c] = pd.to_numeric(all_train_df[c], errors="coerce").fillna(0).astype("float32")
for c in cat_cols:
    all_train_df[c] = all_train_df[c].fillna("UNK").astype(str)

final_pool = Pool(
    data=all_train_df[feature_cols],
    label=all_train_df["label"],
    group_id=all_train_df["user_id"],
    cat_features=cat_cols,
    feature_names=feature_cols,
)

best_iter_a = model_a.get_best_iteration()
best_iter_b = model_b.get_best_iteration()
best_iter_a = params_a["iterations"] if best_iter_a is None or best_iter_a <= 0 else int(best_iter_a * 1.05)
best_iter_b = params_b["iterations"] if best_iter_b is None or best_iter_b <= 0 else int(best_iter_b * 1.05)

final_params_a = params_a.copy()
final_params_b = params_b.copy()
final_params_a["iterations"] = best_iter_a
final_params_b["iterations"] = best_iter_b

final_model_a = CatBoostRanker(**final_params_a)
final_model_b = CatBoostRanker(**final_params_b)

final_model_a.fit(final_pool, verbose=100)
final_model_b.fit(final_pool, verbose=100)

final_model_a.save_model(cfg.output_dir / "catboost_ranker_a.cbm")
final_model_b.save_model(cfg.output_dir / "catboost_ranker_b.cbm")
joblib.dump(feature_cols, cfg.output_dir / "feature_cols.joblib")
joblib.dump(cat_cols, cfg.output_dir / "cat_cols.joblib")

print("Saved models to:", cfg.output_dir)


In [ ]:
full_ref_ts = interactions["event_ts"].max().normalize() + pd.Timedelta(days=1)
full_context = build_context(
    observed=interactions,
    ref_ts=full_ref_ts,
    edition_meta=edition_meta,
    edition_genres=edition_genres,
    users=users,
    cfg=cfg,
)

target_user_ids = targets["user_id"].drop_duplicates().sort_values().to_numpy(dtype=np.int64)
test_candidate_pool, test_sources = build_candidate_pool(full_context, target_user_ids, cfg)
test_features = assemble_candidate_features(full_context, test_candidate_pool, hidden_pairs=None)

for c in num_cols:
    if c in test_features.columns:
        test_features[c] = pd.to_numeric(test_features[c], errors="coerce").fillna(0).astype("float32")
    else:
        test_features[c] = 0.0

for c in cat_cols:
    if c in test_features.columns:
        test_features[c] = test_features[c].fillna("UNK").astype(str)
    else:
        test_features[c] = "UNK"

test_pool = Pool(
    data=test_features[feature_cols],
    cat_features=cat_cols,
    feature_names=feature_cols,
)

test_pred_a = final_model_a.predict(test_pool)
test_pred_b = final_model_b.predict(test_pool)

scored_test = test_features[["user_id", "edition_id", "pre_score"]].copy()
scored_test["pred_a"] = test_pred_a.astype(np.float32)
scored_test["pred_b"] = test_pred_b.astype(np.float32)
wa, wb, wh = blend_ws
scored_test["final_score"] = (wa * scored_test["pred_a"] + wb * scored_test["pred_b"] + wh * scored_test["pre_score"]).astype(np.float32)

print(scored_test.shape)
display(scored_test.head())


In [ ]:
def build_seen_map(seen_pairs: pd.DataFrame) -> dict[int, set[int]]:
    out: dict[int, set[int]] = defaultdict(set)
    for row in seen_pairs.itertuples(index=False):
        out[int(row.user_id)].add(int(row.edition_id))
    return out


def build_user_fallback_lists(
    context: dict[str, Any],
    target_user_ids: np.ndarray,
) -> dict[int, list[int]]:
    user_demo = context["users"][["user_id", "gender", "age_bucket"]]
    demo_pop = context["pop"]["demo_pop"]
    gender_pop = context["pop"]["gender_pop"]
    global_pop = context["pop"]["global_pop"]

    out: dict[int, list[int]] = {}
    demo_map = defaultdict(list)
    gender_map = defaultdict(list)

    for row in demo_pop.sort_values(["gender", "age_bucket", "demo_rank"]).itertuples(index=False):
        demo_map[(int(row.gender), int(row.age_bucket))].append(int(row.edition_id))
    for row in gender_pop.sort_values(["gender", "gender_rank"]).itertuples(index=False):
        gender_map[int(row.gender)].append(int(row.edition_id))

    global_list = global_pop.sort_values("pop_rank")["edition_id"].astype(int).tolist()

    for row in user_demo[user_demo["user_id"].isin(target_user_ids)].itertuples(index=False):
        seq = []
        seq.extend(demo_map.get((int(row.gender), int(row.age_bucket)), []))
        seq.extend(gender_map.get(int(row.gender), []))
        seq.extend(global_list)
        # unique preserve order
        uniq = []
        seen = set()
        for x in seq:
            if x not in seen:
                uniq.append(x)
                seen.add(x)
        out[int(row.user_id)] = uniq

    missing_users = set(int(u) for u in target_user_ids) - set(out.keys())
    for user_id in missing_users:
        out[int(user_id)] = global_list.copy()

    return out


def make_submission(
    scored_df: pd.DataFrame,
    target_users: np.ndarray,
    seen_map: dict[int, set[int]],
    fallback_map: dict[int, list[int]],
    top_k: int = TOP_K,
) -> pd.DataFrame:
    scored_df = scored_df.sort_values(["user_id", "final_score"], ascending=[True, False]).copy()

    submission_rows = []
    grouped = {int(u): grp["edition_id"].astype(int).tolist() for u, grp in scored_df.groupby("user_id")}

    for user_id in tqdm(target_users.astype(int), desc="Build submission"):
        recs: list[int] = []
        used: set[int] = set()
        seen_items = seen_map.get(int(user_id), set())

        for item_id in grouped.get(int(user_id), []):
            if item_id in used or item_id in seen_items:
                continue
            recs.append(int(item_id))
            used.add(int(item_id))
            if len(recs) == top_k:
                break

        if len(recs) < top_k:
            for item_id in fallback_map.get(int(user_id), []):
                if item_id in used or item_id in seen_items:
                    continue
                recs.append(int(item_id))
                used.add(int(item_id))
                if len(recs) == top_k:
                    break

        if len(recs) < top_k:
            global_extra = (
                full_context["pop"]["global_pop"]
                .sort_values("pop_rank")["edition_id"]
                .astype(int).tolist()
            )
            for item_id in global_extra:
                if item_id in used or item_id in seen_items:
                    continue
                recs.append(int(item_id))
                used.add(int(item_id))
                if len(recs) == top_k:
                    break

        if len(recs) != top_k:
            raise RuntimeError(f"Cannot build {top_k} unique items for user {user_id}")

        for rank, edition_id in enumerate(recs, start=1):
            submission_rows.append(
                {
                    "user_id": int(user_id),
                    "edition_id": int(edition_id),
                    "rank": int(rank),
                }
            )

    return pd.DataFrame(submission_rows)


seen_map = build_seen_map(full_context["seen_pairs"])
fallback_map = build_user_fallback_lists(full_context, target_user_ids)

submission = make_submission(
    scored_df=scored_test,
    target_users=target_user_ids,
    seen_map=seen_map,
    fallback_map=fallback_map,
    top_k=TOP_K,
)

ok, errors = validate_submission_rows(
    submission.to_dict(orient="records"),
    target_users=set(int(u) for u in target_user_ids),
    top_k=TOP_K,
)
print("submission valid:", ok)
if not ok:
    print(errors[:10])

submission_path = cfg.output_dir / "submission.csv"
submission.to_csv(submission_path, index=False)
print("saved to:", submission_path)
display(submission.head(30))


## Что обычно оказывается самым сильным в этом пайплайне

На этой задаче топовые фичи почти всегда оказываются из этих блоков:

1. **ALS source rank / ALS pair score / ALS cosine**
2. **KNN similarities** к recent seed items пользователя
3. **recent author / book affinity**
4. **item recent popularity / trend**
5. **source_count и top_source**
6. **user recent activity + item recency**

Что ещё можно дожать поверх этого ноутбука:

- поднять `als_full_topk` / `max_candidates_per_user`
- добавить 3-й ranker (`PairLogitPairwise`) и доблендить
- отдельно обучить ranker на `read-heavy` пользователях и потом смешивать
- сделать text-based item2item по `title + description`
- добавить book-level sequence heuristics для последних событий пользователя
